# Treino da voz da Nair no Colab

Notebook plug and play para fazer fine tuning de uma voz Piper com o dataset preparado da Nair.

Antes de rodar: em Ambiente de execução > Alterar tipo de ambiente de execução, escolha GPU. O ZIP precisa conter metadata.csv sem cabeçalho e a pasta wav/.

In [ ]:
# 1. Configuração
from pathlib import Path
import os, sys, shutil, subprocess, zipfile, csv, json, glob

WORK = Path('/content/nair_piper'); WORK.mkdir(exist_ok=True)
DRIVE_ROOT = Path('/content/drive/MyDrive/PensaoNair')
BATCH_SIZE = 8          # se der CUDA out of memory, mude para 4 ou 2
MAX_EPOCHS = 1000
VOICE_NAME = 'nair'
ESPEAK_VOICE = 'pt-br'
CHECKPOINT_REPO = 'rhasspy/piper-checkpoints'
CHECKPOINT_FILE = 'pt/pt_BR/jeff/medium/epoch=5462-step=118728.ckpt'
print('Configuração pronta')

In [ ]:
# 2. GPU e Piper de treinamento
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ATENÇÃO: nenhuma GPU detectada')
!apt-get -qq update
!apt-get -qq install -y espeak-ng build-essential cmake ninja-build
import subprocess
if not Path('/content/piper1-gpl').exists(): subprocess.run(['git','clone','--depth','1','https://github.com/OHF-Voice/piper1-gpl.git','/content/piper1-gpl'],check=True)
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
%cd /content/nair_piper
!pip -q install huggingface_hub soundfile

In [ ]:
# 3. Drive e upload do dataset
from google.colab import drive, files
drive.mount('/content/drive')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
zips = list(DRIVE_ROOT.glob('nair_dataset*.zip')) + list(Path('/content').glob('nair_dataset*.zip'))
if zips:
    DATA_ZIP = zips[0]
else:
    print('Envie nair_dataset_revisao_final.zip')
    uploaded = files.upload()
    uploaded_name = next(n for n in uploaded if n.lower().endswith('.zip'))
    # files.upload salva no diretório de trabalho atual (que pode ser /content/nair_piper).
    DATA_ZIP = next((p for p in [Path(uploaded_name), Path('/content')/uploaded_name, Path.cwd()/uploaded_name] if p.exists()), None)
    if DATA_ZIP is None:
        found = list(Path('/content').rglob(uploaded_name))
        if not found: raise FileNotFoundError(f'Upload recebido, mas não encontrei {uploaded_name}')
        DATA_ZIP = found[0]
DATA_DIR = WORK/'dataset'
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir()
with zipfile.ZipFile(DATA_ZIP) as z: z.extractall(DATA_DIR)
candidates = [DATA_DIR] + [p for p in DATA_DIR.iterdir() if p.is_dir()]
DATA_ROOT = next((p for p in candidates if (p/'metadata.csv').exists() and (p/'wav').is_dir()), None)
if DATA_ROOT is None: raise FileNotFoundError('ZIP não contém metadata.csv e wav/')
print('Dataset:', DATA_ROOT)

In [ ]:
# 4. Validar e preservar metadados
import soundfile as sf
rows=[]
with (DATA_ROOT/'metadata.csv').open(encoding='utf-8') as f:
    for n,row in enumerate(csv.reader(f,delimiter='|'),1):
        if len(row)!=2: raise ValueError(f'Linha {n} inválida; esperado arquivo|texto')
        wav,text=row; path=DATA_ROOT/'wav'/wav
        if not path.exists(): raise FileNotFoundError(path)
        info=sf.info(path)
        if info.channels != 1 or info.samplerate != 22050: print('Aviso de formato:',wav,info.channels,info.samplerate)
        if not text.strip(): raise ValueError(f'Texto vazio em {wav}')
        rows.append((wav,text))
print(len(rows),'clipes válidos')
shutil.copy2(DATA_ROOT/'metadata.csv',DRIVE_ROOT/'metadata-before-training.csv')

In [ ]:
# 5. Baixar checkpoint base e preparar caminhos
from huggingface_hub import hf_hub_download
CKPT_CACHE=DRIVE_ROOT/'checkpoints'; CKPT_CACHE.mkdir(exist_ok=True)
base_ckpt=hf_hub_download(repo_id=CHECKPOINT_REPO,repo_type='dataset',filename=CHECKPOINT_FILE,local_dir=str(CKPT_CACHE),local_dir_use_symlinks=False)
TRAIN_DIR=DRIVE_ROOT/'training'; TRAIN_DIR.mkdir(parents=True,exist_ok=True)
print('Checkpoint:',base_ckpt)

In [ ]:
# 6. Fine tuning
cmd=[sys.executable,'-m','piper.train','fit',
 '--data.voice_name',VOICE_NAME,'--data.csv_path',str(DATA_ROOT/'metadata.csv'),
 '--data.audio_dir',str(DATA_ROOT/'wav'),'--data.sample_rate','22050',
 '--data.espeak_voice',ESPEAK_VOICE,'--data.cache_dir',str(TRAIN_DIR/'cache'),
 '--data.config_path',str(TRAIN_DIR/'nair.config.json'),'--data.batch_size',str(BATCH_SIZE),
 '--data.validation_split','0.05','--data.num_test_examples','0','--data.num_workers','2',
 '--trainer.max_epochs',str(MAX_EPOCHS),'--trainer.accelerator','gpu','--trainer.devices','1',
 '--trainer.precision','32-true','--trainer.default_root_dir',str(TRAIN_DIR),
 '--ckpt_path',str(sorted(TRAIN_DIR.rglob('*.ckpt'),key=lambda p:p.stat().st_mtime)[-1] if list(TRAIN_DIR.rglob('*.ckpt')) else base_ckpt)]
print(' '.join(cmd))
print('Os checkpoints ficam no Drive. Se a sessão cair, reconecte e execute esta célula; ela retoma o mais recente. Interrompa e reduza BATCH_SIZE se houver OOM.')
subprocess.run(cmd,check=True)

In [ ]:
# 7. Salvar checkpoints no Drive
ckpts=sorted(TRAIN_DIR.rglob('*.ckpt'),key=lambda p:p.stat().st_mtime)
if not ckpts: raise FileNotFoundError('Nenhum checkpoint foi criado')
for p in ckpts:
    shutil.copy2(p,DRIVE_ROOT/'checkpoints'/p.name)
print('Salvos',len(ckpts),'checkpoints no Drive')
final_ckpt=ckpts[-1]

In [ ]:
# 8. Exportar ONNX, testar e baixar pacote
export_dir=DRIVE_ROOT/'export'; export_dir.mkdir(exist_ok=True)
out_onnx=export_dir/'pt_BR-nair-medium.onnx'
subprocess.run([sys.executable,'-m','piper.train.export_onnx','--checkpoint',str(final_ckpt),'--output-file',str(out_onnx)],check=True)
shutil.copy2(TRAIN_DIR/'nair.config.json',export_dir/'pt_BR-nair-medium.onnx.json')
!pip -q install piper-tts
test_wav=export_dir/'nair-teste.wav'
subprocess.run(['piper','--model',str(out_onnx),'--output_file',str(test_wav)],input='Valdir, não deixa essa panela na pia.\n',text=True,check=True)
package=WORK/'pt_BR-nair-medium.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for p in [out_onnx,export_dir/'pt_BR-nair-medium.onnx.json',test_wav]: z.write(p,p.name)
from google.colab import files
files.download(str(package))
print('Pacote exportado:',package)

## Ajustes e continuidade

Se aparecer CUDA out of memory, use BATCH_SIZE = 4 ou 2 e rode o treino de novo. O arquivo .ckpt fica em My Drive/PensaoNair/checkpoints; para continuar uma sessão, troque base_ckpt pelo último checkpoint salvo.

O ponto de partida jeff é um checkpoint masculino em português brasileiro. Ele é usado pela compatibilidade de idioma e qualidade, mas a voz pode precisar de mais treinamento para perder características do Jeff. Compare jeff, faber e cadu se necessário.

Antes do treino, escute revisar.html e corrija metadata.csv se encontrar cortes ou textos errados. Separe algumas frases para avaliação e não use os mesmos clipes no treino e no teste.